# 13 · Post-hoc robustness checks of the main conclusions (exploratory, NOT pre-registered)
**Purpose:** stress-test the conclusions of 10/11 against the weaknesses found in the results review (`04_results_analysis.md`). Everything here is computed from **saved scores only** — no model is trained or run. **Runtime:** CPU, ~10–20 min (B = 2000).

| Block | Question | Output (`09_statistics/posthoc/`) |
|---|---|---|
| A1 | Do RQ1 conclusions survive **seed + document** uncertainty? | `a1_rq1_seedaware.csv` |
| A2 | Do family/script coefficients survive seed resampling and dropping single language pairs? | `a2_rq2_seedaware.csv`, `a2_rq2_leave_pair_out.csv` |
| A3 | LOLO and LOFO (1 seed) compared **seed-matched** with E2/all, against the E2/all seed spread | `a3_lolo_seedmatched.csv`, `a3_lofo_seedmatched.csv` |
| A4 | Low-FPR metrics with CIs (TPR@1%/5%FPR) and threshold transfer at 5% FPR | `a4_lowfpr_main.csv`, `a4_threshold_transfer.csv` |
| B | RQ5 mechanism: does the Kazakh-adapted LM raise the likelihood of **human** text more than of machine text? (tokenizer-neutral bits per character) | `b_memorisation_probe.csv`, `b_bpc_auroc.csv` |
| C | Is Llama-3.3 hard because it is **unseen** or because it is **atypical**? | `c_lofo_relative_error.csv`, `c_generator_descriptors.csv`, `c_generator_atypicality.csv` |
| D | Do fine-tuned and zero-shot detectors complement each other? (validation-calibrated rank ensemble) | `d_ensemble.csv` |
| E | Topic overlap between COLING-kk and our training topics; external AUROC without overlapping topics | `e_coling_topic_overlap.csv` |

**Reporting rule:** results from this notebook go to the paper as *robustness checks / exploratory analyses*, clearly separated from the pre-registered tests in 10.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "13_posthoc_checks"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

## Shared helpers (same conventions as 10)

In [ ]:
import itertools, re, math
B = CFG["stats"]["bootstrap_B"]; ALPHA = CFG["stats"]["alpha"]
OUT = P["stats"] / "posthoc"; OUT.mkdir(parents=True, exist_ok=True)
EVAL = load_eval_texts()
reg = load_registry(); reg = reg[reg.status == "done"].copy(); reg["seed"] = reg.seed.astype(int)
TURKIC = CFG["turkic_targets"]
FT_DETS = [d for d in ["mdeberta", "xlmr", "ngram", "stylo"] if (reg.detector == d).any()]
NEURAL = [d for d in ["mdeberta", "xlmr"] if d in FT_DETS]
FAM = {l: LANG_META[l]["family"] for l in LANGS}; SCR = {l: LANG_META[l]["script"] for l in LANGS}
_cache = {}

def preds(run_id):
    if run_id not in _cache:
        _cache[run_id] = pd.read_parquet(pred_path(run_id)).set_index("doc_id").score
    return _cache[run_id]

def runs(exp, detector, train, heldout="none", seed=None):
    r = reg[(reg.exp == exp) & (reg.detector == detector) & (reg.train == train) & (reg.heldout == heldout)]
    if seed is not None:
        r = r[r.seed == seed]
    return list(r.sort_values("seed").run_id)

def aligned(eval_h, eval_m, lang=None, family=None):
    h = EVAL[(EVAL.eval_set == eval_h) & (EVAL.label == 0)]; m = EVAL[(EVAL.eval_set == eval_m) & (EVAL.label == 1)]
    d = pd.concat([h, m])
    if lang:
        d = d[d.lang == lang]
    if family:
        d = d[(d.label == 0) | (d.family == family)]
    return d.sort_values("orig_doc_id").reset_index(drop=True)

def S_of(run_ids, docs):
    M = np.vstack([preds(r).reindex(docs.doc_id).values for r in run_ids])
    assert not np.isnan(M).any(), "missing scores for some documents"
    return M

def q(x):
    lo, hi = np.quantile(x, [ALPHA / 2, 1 - ALPHA / 2]); return float(lo), float(hi)

def p2(d):
    """Two-sided bootstrap p for H0: parameter = 0."""
    return float(max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / len(d)))

def seed_doc_boot(y, mats, seed=0):
    """Hierarchical bootstrap. mats: list of (k_i, n) score matrices on the SAME docs.
    Documents are resampled once (class-stratified, shared → paired contrasts); seeds are resampled
    with replacement independently for each condition. Returns points (seed-mean AUROC),
    seed+doc draws (len(mats), B) and doc-only draws (len(mats), B)."""
    idx = strat_boot_idx(y, B, seed); rng = np.random.default_rng(seed + 10_007)
    pts, sd, do = [], [], []
    for M in mats:
        A = np.vstack([auroc_boot(y, s, idx) for s in M])            # (k, B)
        k = A.shape[0]; pick = rng.integers(0, k, size=(B, k))
        sd.append(A[pick, np.arange(B)[:, None]].mean(1)); do.append(A.mean(0))
        pts.append(np.mean([auroc(y, s) for s in M]))
    return np.array(pts), np.vstack(sd), np.vstack(do)

log.info(f"done runs={len(reg)} | detectors={sorted(reg.detector.unique())} | B={B}")

## A1 — RQ1 with seed + document uncertainty
The CIs in 10 resample documents only (seed-averaged scores). With 3 seeds per condition, resampling seeds is coarse, so we also report the **sign of all 9 cross-seed contrasts** (each E2/all seed vs each E1 seed). Deterministic detectors (n-gram, stylo) have one "seed", so their two CIs coincide.

In [ ]:
out = []
for det, L in itertools.product(FT_DETS, TURKIC):
    a, b = runs("E2", det, "all"), runs("E1", det, L)
    if not a or not b:
        continue
    d = aligned("test", "test", L); y = d.label.values
    Ma, Mb = S_of(a, d), S_of(b, d)
    pts, sd, do = seed_doc_boot(y, [Ma, Mb], seed=TURKIC.index(L))
    dl = pts[0] - pts[1]; dsd = sd[0] - sd[1]; ddo = do[0] - do[1]
    aa = [auroc(y, s) for s in Ma]; bb = [auroc(y, s) for s in Mb]
    cross = np.array([x - z for x in aa for z in bb])
    out.append({"detector": det, "target": L, "n_seeds_multi": len(a), "n_seeds_mono": len(b), "delta": dl,
                "ci_lo_doc": q(ddo)[0], "ci_hi_doc": q(ddo)[1], "ci_lo_seed_doc": q(dsd)[0], "ci_hi_seed_doc": q(dsd)[1],
                "p_seed_doc": p2(dsd), "cross_seed_pos": int((cross > 0).sum()), "cross_seed_n": len(cross),
                "cross_seed_min": float(cross.min()), "cross_seed_max": float(cross.max()),
                "sd_seed_multi": float(np.std(aa, ddof=1)) if len(aa) > 1 else 0.0, "sd_seed_mono": float(np.std(bb, ddof=1)) if len(bb) > 1 else 0.0})
a1 = pd.DataFrame(out)
if len(a1):
    a1["p_holm_seed_doc"] = holm(a1.p_seed_doc)
a1.to_csv(OUT / "a1_rq1_seedaware.csv", index=False); a1.round(4)

## A2 — RQ2 family vs script: seed resampling and leave-one-pair-out
Same two-way fixed-effects model as 10 (`logit AUROC ~ same_script + same_family + source + target`, 30 off-diagonal pairs), solved by least squares for every bootstrap draw at once. In the seed+doc bootstrap one resampled set of seeds is drawn per **source model** and reused for all its target languages (a seed is one trained model).

In [ ]:
OFF = [(s, t) for s in LANGS for t in LANGS if s != t]

def design(pairs):
    cols = [np.ones(len(pairs)), [int(SCR[s] == SCR[t]) for s, t in pairs], [int(FAM[s] == FAM[t]) for s, t in pairs]]
    for l in LANGS[1:]:
        cols += [[int(s == l) for s, _ in pairs], [int(t == l) for _, t in pairs]]
    return np.column_stack(cols).astype(float)

def logit(a):
    a = np.clip(a, 1e-4, 1 - 1e-4); return np.log(a / (1 - a))

def coefs(pairs, auc):
    """auc: (len(pairs),) or (len(pairs), B) → (2,) or (2, B): [script, family]."""
    beta = np.linalg.pinv(design(pairs)) @ logit(auc)
    return beta[1:3]

rows, lpo = [], []
for det in FT_DETS:
    srcs = [s for s in LANGS if runs("E1", det, s)]
    if len(srcs) < len(LANGS):
        continue
    k = {s: len(runs("E1", det, s)) for s in srcs}
    rng = np.random.default_rng(99); pick = {s: rng.integers(0, k[s], size=(B, k[s])) for s in srcs}
    A_sd, A_do, A_pt, A_seed = {}, {}, {}, {}
    for t in LANGS:
        d = aligned("test", "test", t); y = d.label.values; idx = strat_boot_idx(y, B, LANGS.index(t))
        for s in srcs:
            if s == t:
                continue
            M = S_of(runs("E1", det, s), d)
            A = np.vstack([auroc_boot(y, m, idx) for m in M])
            A_sd[(s, t)] = A[pick[s], np.arange(B)[:, None]].mean(1); A_do[(s, t)] = A.mean(0)
            per_seed = [auroc(y, m) for m in M]; A_pt[(s, t)] = np.mean(per_seed); A_seed[(s, t)] = per_seed
    pt = coefs(OFF, np.array([A_pt[p] for p in OFF]))
    sd = coefs(OFF, np.vstack([A_sd[p] for p in OFF])); do = coefs(OFF, np.vstack([A_do[p] for p in OFF]))
    per_seed = [coefs(OFF, np.array([A_seed[p][i] for p in OFF])) for i in range(min(k.values()))]
    rows.append({"detector": det, "n_seeds": min(k.values()),
                 "coef_script": pt[0], "ci_doc_script": q(do[0]), "ci_seed_doc_script": q(sd[0]),
                 "coef_family": pt[1], "ci_doc_family": q(do[1]), "ci_seed_doc_family": q(sd[1]),
                 "p_family_le_script_seed_doc": float((sd[1] <= sd[0]).mean()),
                 "p_script_le_family_seed_doc": float((sd[0] <= sd[1]).mean()),
                 "per_seed_script": [round(float(c[0]), 3) for c in per_seed], "per_seed_family": [round(float(c[1]), 3) for c in per_seed]})
    # leave-one-unordered-pair-out (both directions removed) + named sensitivity subsets
    subsets = {f"drop {a}<->{b}": [p for p in OFF if set(p) != {a, b}] for a, b in itertools.combinations(LANGS, 2)}
    subsets |= {f"drop {l} entirely": [p for p in OFF if l not in p] for l in LANGS}
    for name, pairs in subsets.items():
        c = coefs(pairs, np.array([A_pt[p] for p in pairs]))
        lpo.append({"detector": det, "subset": name, "n_pairs": len(pairs), "coef_script": float(c[0]), "coef_family": float(c[1]),
                    "family_gt_script": bool(c[1] > c[0])})
a2 = pd.DataFrame(rows)
for c in [c for c in a2.columns if c.startswith("ci_")]:
    a2[c + "_lo"] = a2[c].map(lambda v: v[0]); a2[c + "_hi"] = a2[c].map(lambda v: v[1]); a2 = a2.drop(columns=c)
a2.to_csv(OUT / "a2_rq2_seedaware.csv", index=False)
lpo = pd.DataFrame(lpo); lpo.to_csv(OUT / "a2_rq2_leave_pair_out.csv", index=False)
display_cols = ["detector", "coef_script", "coef_family", "ci_seed_doc_script_lo", "ci_seed_doc_script_hi", "ci_seed_doc_family_lo",
                "ci_seed_doc_family_hi", "p_family_le_script_seed_doc", "per_seed_script", "per_seed_family"]
print(a2[[c for c in display_cols if c in a2]].round(3).to_string())
print(lpo.groupby("detector").agg(script_min=("coef_script", "min"), script_max=("coef_script", "max"),
                                  family_min=("coef_family", "min"), family_max=("coef_family", "max"),
                                  share_family_gt_script=("family_gt_script", "mean")).round(3))

## A3 — single-seed experiments compared seed-matched
LOLO (E2/no-L) and LOFO (E3) were run with seed 13 only. Here they are compared with **E2/all seed 13** (same seed), and the E2/all between-seed SD on the same documents is reported as a yardstick: a |Δ| smaller than ~2 × SD is within ordinary seed noise.

In [ ]:
_single = reg[(reg.exp == "E3") | ((reg.exp == "E2") & reg.train.str.startswith("no-"))]
SEED0 = int(_single.seed.min()) if len(_single) else 13           # the seed used for the single-seed experiments
SEED_OF = reg.set_index("run_id").seed

def seed_matched(a_runs, base_runs_all, d):
    y = d.label.values
    s_a = next((r for r in a_runs if SEED_OF[r] == SEED0), a_runs[0])
    s_b = next((r for r in base_runs_all if SEED_OF[r] == SEED0), base_runs_all[0])
    pts, _, do = seed_doc_boot(y, [S_of([s_a], d), S_of([s_b], d)], seed=3)
    dd = do[0] - do[1]
    sd_all = float(np.std([auroc(y, s) for s in S_of(base_runs_all, d)], ddof=1)) if len(base_runs_all) > 1 else np.nan
    return {"auroc_cond": pts[0], "auroc_base_same_seed": pts[1], "delta_seed_matched": pts[0] - pts[1], "ci_lo": q(dd)[0], "ci_hi": q(dd)[1],
            "p_boot": p2(dd), "sd_base_across_seeds": sd_all,
            "within_seed_noise": bool(abs(pts[0] - pts[1]) < 2 * sd_all) if not np.isnan(sd_all) else None}

out = []
for det, L in itertools.product(FT_DETS, TURKIC):
    a, b = runs("E2", det, f"no-{L}"), runs("E2", det, "all")
    if a and b:
        out.append({"detector": det, "target": L} | seed_matched(a, b, aligned("test", "test", L)))
a3l = pd.DataFrame(out); a3l.to_csv(OUT / "a3_lolo_seedmatched.csv", index=False)
out = []
for det in FT_DETS:
    for f in sorted(set(reg[reg.exp == "E3"].heldout)):
        a, b = runs("E3", det, "all", f), runs("E2", det, "all")
        if a and b:
            out.append({"detector": det, "heldout_family": f} | seed_matched(a, b, aligned("test", "test", None, family=f)))
a3f = pd.DataFrame(out); a3f.to_csv(OUT / "a3_lofo_seedmatched.csv", index=False)
print(a3l.round(4).to_string()); print(a3f.round(4).to_string())

## A4 — low-FPR operating points with uncertainty
**Resolution limit:** with *n* human test texts, FPR can only take values in steps of 1/*n* (≈ 0.5% per language here), so TPR@1%FPR is set by the top ~2 human scores. We therefore report TPR@5%FPR alongside 1%, both with bootstrap CIs (threshold re-estimated in every resample). Threshold transfer is recomputed at a **5%** source-validation FPR (≈ 5 validation humans instead of ~1).

In [ ]:
def tpr_at_fpr_boot(y, s, idx, fpr):
    """Vectorised TPR at the empirical FPR threshold, re-estimated per resample (same rule as threshold_at_fpr)."""
    S = np.asarray(s, float)[idx]; Y = np.asarray(y)[idx]
    negs = -np.sort(-np.where(Y == 0, S, -np.inf), axis=1)
    k = np.floor(fpr * (Y == 0).sum(1)).astype(int)
    thr = negs[np.arange(len(S)), k] + 1e-12
    return ((S > thr[:, None]) & (Y == 1)).sum(1) / (Y == 1).sum(1)

MAIN = {det: runs("E2", det, "all") for det in FT_DETS}
MAIN |= {r.detector: [r.run_id] for r in reg[reg.exp == "E5"].sort_values("detector").itertuples()}
MAIN = {k: v for k, v in MAIN.items() if v}
out = []
for L in LANGS + [None]:
    d = aligned("test", "test", L); y = d.label.values; idx = strat_boot_idx(y, B, 5)
    for det, rids in MAIN.items():
        M = S_of(rids, d)
        for fpr in CFG["stats"]["fpr_points"]:
            bt = np.mean([tpr_at_fpr_boot(y, s, idx, fpr) for s in M], axis=0)
            pt = float(np.mean([tpr_at_fpr(y, s, fpr) for s in M]))
            out.append({"detector": det, "lang": L or "all", "fpr": fpr, "tpr": pt, "ci_lo": q(bt)[0], "ci_hi": q(bt)[1],
                        "n_neg": int((y == 0).sum()), "fpr_resolution": 1 / int((y == 0).sum()), "n_seeds": len(M)})
a4 = pd.DataFrame(out); a4.to_csv(OUT / "a4_lowfpr_main.csv", index=False)
print(a4[a4.lang == "all"].pivot_table(index="detector", columns="fpr", values="tpr").round(3))

TEST_DOCS = {t: aligned("test", "test", t) for t in LANGS}; VAL_DOCS = {s: aligned("val", "val", s) for s in LANGS}
out = []
for det in FT_DETS:
    for s in LANGS:
        for r in runs("E1", det, s):
            sc = preds(r)
            v = VAL_DOCS[s]; vs = sc.reindex(v.doc_id).values
            for fpr in CFG["stats"]["fpr_points"]:
                thr = threshold_at_fpr(v.label.values, vs, fpr)
                for t in LANGS:
                    d = TEST_DOCS[t]; ts = sc.reindex(d.doc_id).values; y = d.label.values
                    out.append({"detector": det, "train": s, "lang": t, "seed": int(SEED_OF[r]), "nominal_fpr": fpr,
                                "n_val_neg": int((v.label == 0).sum()), "realised_fpr": float((ts[y == 0] > thr).mean()),
                                "realised_tpr": float((ts[y == 1] > thr).mean())})
tt = pd.DataFrame(out)
tta = tt.groupby(["detector", "train", "lang", "nominal_fpr"]).agg(realised_fpr=("realised_fpr", "mean"), fpr_seed_min=("realised_fpr", "min"),
                                                                    fpr_seed_max=("realised_fpr", "max"), realised_tpr=("realised_tpr", "mean"),
                                                                    n_seeds=("seed", "nunique"), n_val_neg=("n_val_neg", "first")).reset_index()
tta["ratio_to_nominal"] = tta.realised_fpr / tta.nominal_fpr
tta.to_csv(OUT / "a4_threshold_transfer.csv", index=False)
for fpr in CFG["stats"]["fpr_points"]:
    g = tta[(tta.nominal_fpr == fpr) & (tta.train != tta.lang)]
    print(f"nominal {fpr:.0%}: off-diagonal realised FPR by detector (median / max):")
    print(g.groupby("detector").realised_fpr.agg(["median", "max"]).round(3))

## B — RQ5 mechanism: likelihood shift of human vs machine text under the Kazakh-adapted LM
The two scorers use **different tokenizers**, so per-token log-likelihoods are not comparable. We convert the saved mean per-token log-likelihood (`loglik-*` runs) back to a total and divide by the number of **characters actually predicted** (re-tokenising each text with the exact call used in 08: same tokenizer, truncation at `max_tokens`) → **bits per character (BPC)**, which is tokenizer-neutral.

For each language and class: gain = BPC_generic − BPC_kazakh (positive = the Kazakh model finds the text more probable). **Key statistic: DiD = gain(human) − gain(machine)**, paired document bootstrap.
- Predicted by *memorisation* of Kazakh Wikipedia (the ISSAI model card lists FineWiki among its Kazakh sources): DiD ≫ 0 in kk and in COLING-kk (also Wikipedia), smaller in related languages.
- Also predicted by a *non-memorisation* alternative: the adapted model simply models native human Kazakh better than LLM-generated Kazakh. **This analysis cannot separate the two**; doing so needs non-Wikipedia human Kazakh text or a membership-inference probe on per-token scores (GPU).
- We also report the AUROC of per-character log-likelihood: if per-token normalisation with the new tokenizer caused the collapse of `loglik-kazakh`, the per-character AUROC would recover.

In [ ]:
from transformers import AutoTokenizer
ZS = CFG["zeroshot"]; MAXT = ZS["max_tokens"]
probe_sets = [("test", L) for L in LANGS] + [("ext_coling_norm", "kk")]
docs = pd.concat([aligned(ev, ev, L).assign(eval_name=ev) for ev, L in probe_sets], ignore_index=True)

def char_span(tok, texts, bs=64):
    """Characters predicted (from the 2nd token to the end of the last kept token) and n predicted tokens."""
    chars, ntok = [], []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=MAXT, return_offsets_mapping=True)
        for ids, off in zip(enc["input_ids"], enc["offset_mapping"]):
            off = [o for o in off if o[1] > o[0]]
            chars.append(off[-1][1] - off[1][0] if len(off) > 1 else np.nan); ntok.append(len(ids) - 1)
    return np.array(chars, float), np.array(ntok, float)

bpc = {}
for pair, names in ZS["pairs"].items():
    rid = reg[(reg.exp == "E5") & (reg.detector == f"loglik-{pair}")].run_id
    if rid.empty:
        continue
    tok = AutoTokenizer.from_pretrained(names["observer"])
    nch, ntk = char_span(tok, docs.text.tolist())
    ll = preds(rid.iloc[0]).reindex(docs.doc_id).values                  # mean log-prob per predicted token (nats)
    bpc[pair] = -(ll * ntk) / (nch * math.log(2))
    docs[f"ntok_{pair}"] = ntk; docs[f"nchar_{pair}"] = nch
    log.info(f"[{pair}] tokens/char median {np.nanmedian(ntk / nch):.3f}; truncated share {(ntk >= MAXT - 1).mean():.3f}")
assert {"generic", "kazakh"} <= set(bpc), "needs loglik-generic and loglik-kazakh runs"
docs["bpc_generic"], docs["bpc_kazakh"] = bpc["generic"], bpc["kazakh"]
docs["gain"] = docs.bpc_generic - docs.bpc_kazakh
docs = docs[np.isfinite(docs.gain)].reset_index(drop=True)

out, aur = [], []
for (ev, L), g in docs.groupby(["eval_name", "lang"]):
    y = g.label.values; gain = g.gain.values; idx = strat_boot_idx(y, B, 11)
    G = gain[idx]; Y = y[idx]
    gh = (G * (Y == 0)).sum(1) / (Y == 0).sum(1); gm = (G * (Y == 1)).sum(1) / (Y == 1).sum(1); did = gh - gm
    row = {"eval": ev, "lang": L, "n": len(g)}
    for cls, lab in (("human", 0), ("machine", 1)):
        h = g[g.label == lab]
        row |= {f"bpc_generic_{cls}": h.bpc_generic.mean(), f"bpc_kazakh_{cls}": h.bpc_kazakh.mean(), f"gain_{cls}": h.gain.mean()}
    row |= {"did_human_minus_machine": float(gain[y == 0].mean() - gain[y == 1].mean()), "did_ci_lo": q(did)[0], "did_ci_hi": q(did)[1],
            "did_p_boot": p2(did)}
    out.append(row)
    for pair in ("generic", "kazakh"):
        s = -g[f"bpc_{pair}"].values                                   # higher likelihood → machine (as loglik detector)
        a, lo, hi = boot_ci_auroc(y, s[None, :], B=B, seed=12)
        aur.append({"eval": ev, "lang": L, "scorer": pair, "auroc_per_char_loglik": a, "ci_lo": lo, "ci_hi": hi})
b = pd.DataFrame(out); b.to_csv(OUT / "b_memorisation_probe.csv", index=False)
ba = pd.DataFrame(aur).pivot_table(index=["eval", "lang"], columns="scorer", values="auroc_per_char_loglik").reset_index()
pd.DataFrame(aur).to_csv(OUT / "b_bpc_auroc.csv", index=False)
print(b[["eval", "lang", "gain_human", "gain_machine", "did_human_minus_machine", "did_ci_lo", "did_ci_hi"]].round(3).to_string())
print(ba.round(3).to_string())

## C — Llama-3.3: unseen-family penalty or intrinsic difficulty?
**C1 Relative error ratio** RER = (1 − AUROC_unseen) / (1 − AUROC_seen), seed-matched (E3 vs E2/all, seed 13), pooled over languages, document bootstrap. If RER for Meta is similar to the other families, its larger absolute ΔAUROC reflects its higher base difficulty rather than a larger unseen-family penalty.
**C2 Descriptors** of each generator's texts (all splits) and **C3 atypicality**: cross-validated AUROC of a logistic regression separating one generator's texts from the other generators' texts using only these descriptors (+ language indicators).

In [ ]:
out = []
for det in FT_DETS:
    for f in sorted(set(reg[reg.exp == "E3"].heldout)):
        a, b_ = runs("E3", det, "all", f, seed=SEED0) or runs("E3", det, "all", f), runs("E2", det, "all", seed=SEED0) or runs("E2", det, "all")
        if not a or not b_:
            continue
        d = aligned("test", "test", None, family=f); y = d.label.values; idx = strat_boot_idx(y, B, 21)
        au = auroc_boot(y, S_of(a[:1], d)[0], idx); as_ = auroc_boot(y, S_of(b_[:1], d)[0], idx)
        pu, ps = auroc(y, S_of(a[:1], d)[0]), auroc(y, S_of(b_[:1], d)[0])
        rer = (1 - au) / np.clip(1 - as_, 1e-6, None)
        out.append({"detector": det, "heldout_family": f, "auroc_unseen": pu, "auroc_seen": ps, "err_seen": 1 - ps, "err_unseen": 1 - pu,
                    "rer": (1 - pu) / max(1 - ps, 1e-6), "rer_ci_lo": q(rer)[0], "rer_ci_hi": q(rer)[1]})
c1 = pd.DataFrame(out); c1.to_csv(OUT / "c_lofo_relative_error.csv", index=False)
print(c1.pivot_table(index="detector", columns="heldout_family", values="rer").round(2))

ds = load_dataset()
LAT = re.compile(r"[A-Za-z]"); CYR = re.compile(r"[Ѐ-ӿ]"); SENT = re.compile(r"(?<=[.!?…])\s+")
def descriptors(t, lang):
    w = t.split(); n = max(len(w), 1)
    sents = [s for s in SENT.split(t.strip()) if s]
    win = 50; ttr = np.mean([len(set(w[i:i + win])) / win for i in range(0, max(1, n - win + 1), 10)]) if n >= win else len(set(w)) / n
    tri = [" ".join(w[i:i + 3]) for i in range(max(0, n - 2))]
    letters = sum(c.isalpha() for c in t) or 1
    foreign = len(LAT.findall(t)) if SCR[lang] == "Cyrillic" else len(CYR.findall(t))
    return {"n_words": n, "mean_sent_words": n / max(len(sents), 1), "mattr50": ttr, "distinct3": len(set(tri)) / max(len(tri), 1),
            "punct_per_word": sum(not c.isalnum() and not c.isspace() for c in t) / n, "digit_share": sum(c.isdigit() for c in t) / max(len(t), 1),
            "foreign_script_share": foreign / letters, "comma_per_word": t.count(",") / n}
desc = pd.DataFrame([descriptors(t, l) for t, l in zip(ds.text, ds.lang)])
dd = pd.concat([ds[["lang", "label", "generator", "family"]].reset_index(drop=True), desc], axis=1)
FEATS = list(desc.columns)
c2 = dd.groupby(["generator"])[FEATS].mean().round(4)
# standardised difference from human texts of the same language, averaged over languages
z = []
for (l, gname), g in dd[dd.label == 1].groupby(["lang", "generator"]):
    h = dd[(dd.lang == l) & (dd.label == 0)]
    z.append({"lang": l, "generator": gname} | {f: (g[f].mean() - h[f].mean()) / (h[f].std(ddof=1) + 1e-9) for f in FEATS})
c2z = pd.DataFrame(z).groupby("generator")[FEATS].mean().round(3)
yield_path = P["validated"] / "generation_yield.csv"
if yield_path.exists():
    gy = pd.read_csv(yield_path).groupby("generator")[["n", "valid", "lid_fail", "short"]].sum()
    gy["invalid_rate"] = 1 - gy.valid / gy.n
    c2 = c2.join(gy)
c2.join(c2z, rsuffix="_z_vs_human").to_csv(OUT / "c_generator_descriptors.csv")
print(c2.round(3).to_string()); print("standardised difference vs human:\n", c2z.to_string())

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
mm = dd[dd.label == 1].reset_index(drop=True)
X = pd.concat([mm[FEATS], pd.get_dummies(mm.lang, prefix="lang", dtype=float)], axis=1).values
out = []
for gname in sorted(mm.generator.unique()):
    yy = (mm.generator == gname).astype(int).values
    pr = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X, yy,
                           cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    a, lo, hi = boot_ci_auroc(yy, pr[None, :], B=B, seed=31)
    out.append({"generator": gname, "cv_auroc_vs_other_generators": a, "ci_lo": lo, "ci_hi": hi, "n": int(yy.sum())})
c3 = pd.DataFrame(out); c3.to_csv(OUT / "c_generator_atypicality.csv", index=False); c3.round(3)

## D — complementarity: validation-calibrated rank ensemble (exploratory)
Each member's score is mapped to its empirical CDF on the **pooled validation split** (all languages, both classes) — a deployable calibration that uses no test data; multi-seed members average their seeds' CDF values; the ensemble is the mean over members. Reported on clean test, attacked sets and COLING-kk. Δ = ensemble − its best member on the clean pooled test (paired document bootstrap).

In [ ]:
ATT = [("test", "test")] + [("test", n) for n in ("T1_para", "T2_bt", "T3_h05", "T3_h15", "T4_t05", "T4_t15")] + \
      [(n + "both", n) for n in ("T3_h05", "T3_h15", "T4_t05", "T4_t15")] + [("ext_coling_norm", "ext_coling_norm")]
ATT = [(h, m) for h, m in ATT if (EVAL.eval_set == h).any() and (EVAL.eval_set == m).any()]
VAL = EVAL[EVAL.eval_set == "val"]

def ecdf_scores(run_ids, docs):
    out = []
    for r in run_ids:
        ref = np.sort(preds(r).reindex(VAL.doc_id).dropna().values)
        out.append(np.searchsorted(ref, preds(r).reindex(docs.doc_id).values, side="right") / len(ref))
    return np.mean(out, axis=0)

ZS_MAIN = "binoculars-generic" if "binoculars-generic" in MAIN else None
COMBOS = {"mDeBERTa+Binoculars": ["mdeberta", ZS_MAIN], "mDeBERTa+ngram": ["mdeberta", "ngram"],
          "mDeBERTa+XLM-R+ngram+Binoculars": ["mdeberta", "xlmr", "ngram", ZS_MAIN]}
COMBOS = {k: v for k, v in COMBOS.items() if all(m in MAIN for m in v if m) and None not in v}
clean = aligned("test", "test"); yc = clean.label.values
best_member = {k: max(v, key=lambda m: np.mean([auroc(yc, s) for s in S_of(MAIN[m], clean)])) for k, v in COMBOS.items()}
out = []
for (h_set, m_set), L in itertools.product(ATT, [None] + LANGS):
    d = aligned(h_set, m_set, L)
    if d.empty or d.label.nunique() < 2:
        continue
    y = d.label.values; idx = strat_boot_idx(y, B, 41)
    mem = {m: ecdf_scores(MAIN[m], d) for m in set(sum(COMBOS.values(), []))}
    for name, members in COMBOS.items():
        ens = np.mean([mem[m] for m in members], axis=0); bm = best_member[name]
        be, bb = auroc_boot(y, ens, idx), auroc_boot(y, mem[bm], idx)
        out.append({"eval": m_set if h_set == "test" else h_set, "lang": L or "all", "ensemble": name, "best_member": bm, "auroc_ensemble": auroc(y, ens),
                    "auroc_best_member": auroc(y, mem[bm]), "delta": auroc(y, ens) - auroc(y, mem[bm]),
                    "ci_lo": q(be - bb)[0], "ci_hi": q(be - bb)[1], "p_boot": p2(be - bb),
                    "tpr5_ensemble": tpr_at_fpr(y, ens, 0.05), "tpr5_best_member": tpr_at_fpr(y, mem[bm], 0.05)})
dE = pd.DataFrame(out); dE.to_csv(OUT / "d_ensemble.csv", index=False)
print(dE[dE.lang == "all"].pivot_table(index="eval", columns="ensemble", values="delta").round(4))

## E — COLING-kk topic overlap with our training topics
COLING-kk prompts have the form "…write one paragraph about the following topic in Kazakh: {topic}." The topic is compared (case-folded, parenthetical qualifiers removed) with the titles of our Kazakh articles. Overlap is not label leakage (different texts, near-duplicates were removed in 04), but a topic seen in training can make the external test easier, so we re-evaluate on non-overlapping topics.

In [ ]:
def norm_title(t):
    return re.sub(r"\s*\(.*?\)\s*$", "", str(t)).strip().casefold()
raw = pd.DataFrame(read_jsonl(P["raw_external"] / "coling2025_kazakh.jsonl"))
raw["topic"] = raw.prompt.astype(str).str.extract(r":\s*(.+?)\.?\s*$")[0].map(norm_title)
kk = ds[ds.lang == "kk"]
titles = {sp: set(kk[kk.split == sp].title.map(norm_title)) for sp in ("train", "val", "test")}
raw["in_our_train"] = raw.topic.isin(titles["train"]); raw["in_our_any"] = raw.topic.isin(set().union(*titles.values()))
overlap_ids = set("colingkk-" + raw[raw.in_our_train].testset_id.astype(str))
summary = {"coling_rows": len(raw), "coling_topics": raw.topic.nunique(), "topics_in_our_kk_train": int(raw[raw.in_our_train].topic.nunique()),
           "rows_in_our_kk_train_topics": int(raw.in_our_train.sum()), "topics_in_our_kk_any_split": int(raw[raw.in_our_any].topic.nunique())}
log.info(summary)
out = []
d_all = aligned("ext_coling_norm", "ext_coling_norm"); d_no = d_all[~d_all.orig_doc_id.isin(overlap_ids)].reset_index(drop=True)
for det in FT_DETS:
    for name, rids in {"E2_all": runs("E2", det, "all"), "E3_no_openai": runs("E3", det, "all", "openai"), "E1_kk": runs("E1", det, "kk")}.items():
        if not rids:
            continue
        for subset, d in (("all_topics", d_all), ("non_overlapping_topics", d_no)):
            if d.label.nunique() < 2:
                continue
            a, lo, hi = boot_ci_auroc(d.label.values, S_of(rids, d), B=B, seed=51)
            out.append({"detector": det, "model": name, "subset": subset, "n": len(d), "auroc": a, "ci_lo": lo, "ci_hi": hi} | summary)
e = pd.DataFrame(out); e.to_csv(OUT / "e_coling_topic_overlap.csv", index=False)
e.pivot_table(index=["detector", "model"], columns="subset", values="auroc").round(4)

## Output manifest

In [ ]:
files = sorted(f for f in OUT.glob("*.csv") if not f.name.startswith("_"))
man = pd.DataFrame({"file": [f.name for f in files], "rows": [len(pd.read_csv(f)) for f in files]})
man.to_csv(OUT / "_manifest.csv", index=False); log.info(f"wrote {len(files)} files to {OUT}"); man